# Context Model Training from Raw `.osz` (API)

This notebook trains only the context model using `train_api` (no CLI).
It keeps the same main settings as `train_context_raw_data.ipynb` and supports resume from `last.ckpt`.

In [1]:
from pathlib import Path

import torch

from src.model import (
    ArchitectureSpec,
    TrainingSpec,
    WandbConfig,
    build_training_artifacts,
    create_training_context,
    load_training_context_from_checkpoint,
    prepare_sample_data_artifacts,
    train_context,
)


In [ ]:
# Same intent/defaults as the CLI notebook
raw_osz_dir = Path("E:/batchbeatmapdownloadtesttemp")
cache_root = Path("C:/taiko-transformer-cache")
data_root = cache_root / "batchbeatmapdownloadtest"
training_dir = data_root / "training"
repo_root = Path.cwd()
checkpoints_dir = repo_root / "checkpoints" / "context"

epochs = 10
batch_size = 32
num_workers = 8

# Aggressive speed-first context budget.
history_max_tokens = 128
retrieval_top_k = 1
retrieval_max_tokens_per_window = 12
retrieval_exclude_last_n_windows = 2
use_motif_retrieval = True
max_cached_charts = 2

# Runtime acceleration knobs.
precision = "auto"
pin_memory = True
persistent_workers = True
prefetch_factor = 4
architecture_name = "taiko_context_transformer"
run_name = "test_10epochs"
prepare_data = False
save_inference_every_n_steps = 1000
use_resume_if_available = True
use_wandb = False
wandb_log_every_batches = 100
wandb_notebook_name = "train_context_raw_data_api.ipynb"
wandb_api_key = ""
wandb_offline = False

if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

checkpoints_dir.mkdir(parents=True, exist_ok=True)
last_checkpoint = checkpoints_dir / "last.ckpt"

index_cache_dir = training_dir / "index_cache"
inference_snapshots_dir = checkpoints_dir / "inference_snapshots"

print(f"raw_osz_dir={raw_osz_dir}")
print(f"data_root={data_root}")
print(f"checkpoints_dir={checkpoints_dir}")
print(f"last_checkpoint={last_checkpoint}")
print(f"index_cache_dir={index_cache_dir}")
print(f"inference_snapshots_dir={inference_snapshots_dir}")
print(f"device={device}")


raw_osz_dir=E:\batchbeatmapdownloadtesttemp
data_root=E:\batchbeatmapdownloadtest
checkpoints_dir=c:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\context
last_checkpoint=c:\Users\28548\PythonNotebooks\taiko-diffusion\checkpoints\context\last.ckpt
device=cuda


In [3]:
# Prepare data artifacts from raw .osz inputs.
if prepare_data:
    artifacts = prepare_sample_data_artifacts(
        osz_inputs=[str(raw_osz_dir)],
        data_root=data_root,
    )
else:
    artifacts = build_training_artifacts(data_root, checkpoints_dir=checkpoints_dir)
    print("Skipping raw-data preparation and reusing existing training artifacts.")
print(artifacts)

architecture_spec = ArchitectureSpec(
    name=architecture_name,
    history_max_tokens=history_max_tokens,
    retrieval_top_k=retrieval_top_k,
    retrieval_max_tokens_per_window=retrieval_max_tokens_per_window,
    retrieval_exclude_last_n_windows=retrieval_exclude_last_n_windows,
    use_motif_retrieval=use_motif_retrieval,
    max_cached_charts=max_cached_charts,
)
training_spec = TrainingSpec(
    epochs=epochs,
    batch_size=batch_size,
    num_workers=num_workers,
    device=device,
    precision=precision,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=prefetch_factor,
)

wandb_config = None
if use_wandb:
    wandb_config = WandbConfig(
        enabled=True,
        run_name=run_name,
        log_every_n_batches=wandb_log_every_batches,
        notebook_name=wandb_notebook_name,
        offline=wandb_offline,
        api_key=wandb_api_key,
        mode_name_for_run=architecture_name,
    )


Unpacking .osz files (total):  33%|███▎      | 3272/10047 [00:01<00:04, 1554.90file/s]

[WARN] Skipping corrupted/unreadable archive: E:\batchbeatmapdownloadtesttemp\1804215.osz (File is not a zip file)


Unpacking .osz files (total):  46%|████▌     | 4580/10047 [00:02<00:06, 812.52file/s] 

[WARN] Skipping corrupted/unreadable archive: E:\batchbeatmapdownloadtesttemp\2057179.osz (File is not a zip file)


Unpacking .osz files (total):  54%|█████▍    | 5465/10047 [00:03<00:06, 720.96file/s]

...

[WARN] Skipping corrupted/unreadable archive: E:\batchbeatmapdownloadtesttemp\2393065.osz (File is not a zip file)


Unpacking .osz files (total): 100%|██████████| 10047/10047 [00:13<00:00, 752.80file/s]


Unpack summary | total: 10047 | unpacked: 0 | skipped existing: 10024 | failed corrupt: 23
Skipped 23 corrupted/unreadable archive(s).
[INFO] Fast-skip chart before full parse: E:\batchbeatmapdownloadtest\unpacked\100019\Owl City & Carly Rae Jepsen - Good Time (Gero) [Easy].osu (non_taiko_mode_0)
[INFO] Fast-skip chart before full parse: E:\batchbeatmapdownloadtest\unpacked\100019\Owl City & Carly Rae Jepsen - Good Time (Gero) [ezek's Normal].osu (non_taiko_mode_0)
[INFO] Fast-skip chart before full parse: E:\batchbeatmapdownloadtest\unpacked\100019\Owl City & Carly Rae Jepsen - Good Time (Gero) [Hard].osu (non_taiko_mode_0)
[INFO] Fast-skip chart before full parse: E:\batchbeatmapdownloadtest\unpacked\100019\Owl City & Carly Rae Jepsen - Good Time (Gero) [Natsu's Insane].osu (non_taiko_mode_0)
[INFO] Fast-skip chart before full parse: E:\batchbeatmapdownloadtest\unpacked\100019\Owl City & Carly Rae Jepsen - Good Time (Gero) [Normal].osu (non_taiko_mode_0)
[INFO] Fast-skip chart before

c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,
c:\Users\28548\PythonNotebooks\taiko-diffusion\src\preprocessing\beat_aligned_dataset.py:585: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=None, mono=True)
c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


TrainingArtifacts(data_root=WindowsPath('E:/batchbeatmapdownloadtest'), audio_dir=WindowsPath('E:/batchbeatmapdownloadtest/beat_aligned_dataset/audio_npz'), token_dir=WindowsPath('E:/batchbeatmapdownloadtest/beat_aligned_dataset/token_json'), chart_metadata_csv=WindowsPath('E:/batchbeatmapdownloadtest/chart_index/chart_build_summary.csv'), sequence_metadata_csv=WindowsPath('E:/batchbeatmapdownloadtest/beat_aligned_dataset/sequence_metadata.csv'), training_dir=WindowsPath('E:/batchbeatmapdownloadtest/training'), splits_json=WindowsPath('E:/batchbeatmapdownloadtest/training/splits.json'), vocab_json=WindowsPath('E:/batchbeatmapdownloadtest/training/vocab.json'), checkpoints_dir=WindowsPath('E:/batchbeatmapdownloadtest/training/checkpoints'))


In [4]:
if use_resume_if_available and last_checkpoint.exists():
    context = load_training_context_from_checkpoint(
        last_checkpoint,
        data_root=data_root,
        device=device,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
        history_max_tokens=history_max_tokens,
        retrieval_top_k=retrieval_top_k,
        retrieval_max_tokens_per_window=retrieval_max_tokens_per_window,
        retrieval_exclude_last_n_windows=retrieval_exclude_last_n_windows,
        use_motif_retrieval=use_motif_retrieval,
        max_cached_charts=max_cached_charts,
        precision=precision,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    print(f"Resuming from checkpoint: {last_checkpoint}")
else:
    context = create_training_context(
        data_root=data_root,
        architecture_spec=architecture_spec,
        training_spec=training_spec,
        checkpoints_dir=checkpoints_dir,
        index_cache_dir=index_cache_dir,
        history_max_tokens=history_max_tokens,
        retrieval_top_k=retrieval_top_k,
        retrieval_max_tokens_per_window=retrieval_max_tokens_per_window,
        retrieval_exclude_last_n_windows=retrieval_exclude_last_n_windows,
        use_motif_retrieval=use_motif_retrieval,
        max_cached_charts=max_cached_charts,
        precision=precision,
        pin_memory=pin_memory,
        persistent_workers=persistent_workers,
        prefetch_factor=prefetch_factor,
    )
    print("Starting a fresh context-model training run.")

print(f"start_epoch={context.start_epoch}")
print(f"target_epochs={epochs}")
print(f"architecture={context.architecture_spec.name}")


Starting a fresh context-model training run.
start_epoch=1
target_epochs=10
architecture=taiko_context_transformer


In [5]:
context = train_context(
    context,
    epochs=epochs,
    log_every_n_batches=wandb_log_every_batches,
    wandb_config=wandb_config,
    save_inference_every_n_steps=save_inference_every_n_steps,
    inference_snapshots_dir=inference_snapshots_dir,
)

print("Training finished.")
print(f"last checkpoint: {(checkpoints_dir / 'last.ckpt').resolve()}")
print(f"best checkpoint: {(checkpoints_dir / 'best.ckpt').resolve()}")
print(f"inference snapshots dir: {inference_snapshots_dir.resolve()}")


Training:   0%|          | 0/140022 [00:00<?, ?it/s]c:\Users\28548\.conda\envs\pytorch\Lib\site-packages\torch\nn\functional.py:5476: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(q, k, v, attn_mask, dropout_p, is_causal)


MemoryError: Unable to allocate 36.1 MiB for an array with shape (9461760,) and data type float32